In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
# with open('note.json', 'r') as file:
#     info = json.loads(file.read())

In [3]:
# info

In [5]:
df = pd.read_parquet('../data/aircraft engine/PM_train.parquet')

In [6]:
df.RUL

0        191
1        190
2        189
3        188
4        187
        ... 
20626      4
20627      3
20628      2
20629      1
20630      0
Name: RUL, Length: 20631, dtype: int64

In [7]:
df['RULC'] = df['RUL'].map(lambda x: 1 if x<30 else 0)

In [8]:
df['RULC']

0        0
1        0
2        0
3        0
4        0
        ..
20626    1
20627    1
20628    1
20629    1
20630    1
Name: RULC, Length: 20631, dtype: int64

In [9]:
df['RULC'].value_counts()[0]/df['RULC'].value_counts()[1]

5.877

In [10]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

In [11]:
from sklearn.linear_model import LogisticRegressionCV

In [12]:
from sklearn.svm import SVC

In [13]:
import xgboost as xgb

In [18]:
from scipy.stats import loguniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, accuracy_score
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [19]:
df.head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s15,s16,s17,s18,s19,s20,s21,max,RUL,RULC
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,191,0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,190,0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,189,0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,188,0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,187,0


In [20]:
x = df.drop(['cycle','max','RUL','RULC'], axis=1)
y = df['RULC']

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
x_temp, x_val, y_temp, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_temp = scaler.transform(x_temp)
x_val = scaler.transform(x_val)

In [22]:
def train_test(x,y):
    x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.21, random_state=42)
    scaler = MinMaxScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)
    return x_train_scaled,x_test_scaled,y_train,y_test

In [24]:
random_forest = RandomForestClassifier(random_state=42, class_weight={0:1, 1:7})
gradient_boosting = GradientBoostingClassifier(random_state=42 , n_iter_no_change=10)
logistic = LogisticRegressionCV(random_state=42, Cs=11, cv=3, scoring='accuracy', solver='saga', class_weight={0:1, 1:7}, l1_ratios=[0, 0.5, 1])
svc = SVC()
callback = [xgb.callback.EarlyStopping(rounds=10)]
xtreme_gb = xgb.XGBClassifier(objective='binary:logistic', random_state=42, scale_pos_weight=7, callbacks=callback, eval_metric='aucpr')

In [25]:
rf_p = {
    'n_estimators':np.logspace(4, 5, 5, base=5, dtype=int),
    'criterion':['gini', "entropy", "log_loss"],
    'max_depth':[None, 3, 5, 7, 10],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_features':['sqrt', 'log2'],
    'bootstrap':[True, False],
    # 'max_samples':[0.8, 1.0],
}

gb_p = {
    'learning_rate':loguniform(1e-4, 100),
    'n_estimators':np.logspace(4, 5, 5, base=5, dtype=int),
    'subsample':[1.0, 0.8, 0.6],
    'criterion':['friedman_mse','squared_error'],
    'max_features':['sqrt', 'log2'],
    'min_samples_split':[2, 3, 4, 5, 6, 7, 9],
    'min_samples_leaf':[1, 2, 3, 4, 5],
    'max_depth':[3, 5, 7, 10],
    # tol=0.0001,
    # validation_fraction=0.1,
}

log_p = {
    # 'Cs':11,
    # dual=False, # False if sample>feature
    'penalty':['l2', 'l1', 'elasticnet'],
    # tol=0.0001,
    'max_iter':[100, 200, 300, 500, 1000]

}

svc_p = {
    
}

xgb_p = {
    'n_estimators': np.logspace(4, 5, 5, base=5, dtype=int),               # Set high for early stopping
    'learning_rate': loguniform(1e-4, 100),  # Step size shrinkage
    'max_depth': [None, 3, 5, 7, 10],              # Tree complexity
    'subsample': [0.8, 1.0],             # Rows per tree
    'colsample_bytree': [0.8, 1.0],      # Columns per tree
    'grow_policy': ['depthwise', 'lossguide'],
    # Regularization
    'reg_alpha': [0, 0.1, 1, 5],
    'reg_lambda': [0.1, 1, 10, 20],
    'gamma': [0, 0.1, 1],
    # unbalanced
    # 'min_child_weight': [1, 5, 10],
    # 'eval_metric': ['logloss', 'aucpr']  # Handle class imbalance
}

In [26]:
grid_rf = RandomizedSearchCV(estimator=random_forest, param_distributions=rf_p, cv=3, n_iter=2, random_state=42, scoring='accuracy', verbose=2)
grid_gb = RandomizedSearchCV(estimator=gradient_boosting, param_distributions=gb_p, cv=3, n_iter=2, random_state=42, scoring='accuracy', verbose=2)

In [27]:
grid_log = RandomizedSearchCV(estimator=logistic, param_distributions=log_p, cv=3, n_iter=2, random_state=42, scoring='accuracy', verbose=2)
grid_xgb = RandomizedSearchCV(estimator=xtreme_gb, param_distributions=xgb_p, cv=3, n_iter=2, random_state=42, scoring='accuracy', verbose=2)

In [28]:
grid_rf.fit(x_train, y_train)
grid_gb.fit(x_train, y_train)

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV] END bootstrap=False, criterion=entropy, max_depth=5, max_features=log2, max_samples=0.8, min_samples_leaf=2, min_samples_split=4, n_estimators=625; total time=   0.0s
[CV] END bootstrap=False, criterion=entropy, max_depth=5, max_features=log2, max_samples=0.8, min_samples_leaf=2, min_samples_split=4, n_estimators=625; total time=   0.0s
[CV] END bootstrap=False, criterion=entropy, max_depth=5, max_features=log2, max_samples=0.8, min_samples_leaf=2, min_samples_split=4, n_estimators=625; total time=   0.0s
[CV] END bootstrap=True, criterion=gini, max_depth=3, max_features=sqrt, max_samples=0.8, min_samples_leaf=5, min_samples_split=6, n_estimators=625; total time=   3.2s
[CV] END bootstrap=True, criterion=gini, max_depth=3, max_features=sqrt, max_samples=0.8, min_samples_leaf=5, min_samples_split=6, n_estimators=625; total time=   2.7s
[CV] END bootstrap=True, criterion=gini, max_depth=3, max_features=sqrt, max_samples=0.8,

/home/harsh/lab/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
3 fits failed out of a total of 6.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/home/harsh/lab/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/harsh/lab/lib/python3.11/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/harsh/lab/lib/python3.11/site-packages/sklearn/ensemble/_forest.py", line 433, in fit
    raise ValueE

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV] END criterion=friedman_mse, learning_rate=6.015308718396457, max_depth=7, max_features=sqrt, min_samples_leaf=5, min_samples_split=6, n_estimators=934, subsample=0.6; total time=   0.1s
[CV] END criterion=friedman_mse, learning_rate=6.015308718396457, max_depth=7, max_features=sqrt, min_samples_leaf=5, min_samples_split=6, n_estimators=934, subsample=0.6; total time=   0.1s
[CV] END criterion=friedman_mse, learning_rate=6.015308718396457, max_depth=7, max_features=sqrt, min_samples_leaf=5, min_samples_split=6, n_estimators=934, subsample=0.6; total time=   0.1s
[CV] END criterion=friedman_mse, learning_rate=0.00022310108018679258, max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=4, n_estimators=3125, subsample=0.8; total time=  58.0s
[CV] END criterion=friedman_mse, learning_rate=0.00022310108018679258, max_depth=10, max_features=sqrt, min_samples_leaf=4, min_samples_split=4, n_estimators=3125, subsam

RandomizedSearchCV(cv=3,
                   estimator=GradientBoostingClassifier(n_iter_no_change=10,
                                                        random_state=42),
                   n_iter=2,
                   param_distributions={'criterion': ['friedman_mse',
                                                      'squared_error'],
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fbba01d2110>,
                                        'max_depth': [3, 5, 7, 10],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 3, 4, 5],
                                        'min_samples_split': [2, 3, 4, 5, 6, 7,
                                                              9],
                                        'n_estimators': array([ 625,  934, 1397, 2089, 3125]),
                                        'subsample': [1.0, 0.8, 0.6]},
                   random_state=42, scoring='accuracy', verbose=2)

In [29]:
grid_log.fit(x_train, y_train)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1861: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ...........................max_iter=500, penalty=l2; total time=   9.5s


/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1861: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ...........................max_iter=500, penalty=l2; total time=   9.6s


...

[CV] END ...................max_iter=500, penalty=elasticnet; total time=  29.1s


/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ...................max_iter=500, penalty=elasticnet; total time=  21.2s


/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1861: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/harsh/lab/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


RandomizedSearchCV(cv=3,
                   estimator=LogisticRegressionCV(Cs=11,
                                                  class_weight={0: 1, 1: 7},
                                                  cv=3, l1_ratios=[0, 0.5, 1],
                                                  random_state=42,
                                                  scoring='accuracy',
                                                  solver='saga'),
                   n_iter=2,
                   param_distributions={'max_iter': [100, 200, 300, 500, 1000],
                                        'penalty': ['l2', 'l1', 'elasticnet']},
                   random_state=42, scoring='accuracy', verbose=2)

In [31]:
grid_xgb.fit(x_temp, y_temp, eval_set=[(x_val, y_val)])

Fitting 3 folds for each of 2 candidates, totalling 6 fits
[0]	validation_0-aucpr:0.83570
[1]	validation_0-aucpr:0.53838
[2]	validation_0-aucpr:0.67086
[3]	validation_0-aucpr:0.67790
[4]	validation_0-aucpr:0.60265
[5]	validation_0-aucpr:0.73753
[6]	validation_0-aucpr:0.68879
[7]	validation_0-aucpr:0.66595
[8]	validation_0-aucpr:0.72945
[9]	validation_0-aucpr:0.64747
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=depthwise, learning_rate=2.465832945854912, max_depth=10, n_estimators=3125, reg_alpha=1, reg_lambda=1, subsample=0.8; total time=   0.2s
[0]	validation_0-aucpr:0.84276
[1]	validation_0-aucpr:0.46084
[2]	validation_0-aucpr:0.63434
[3]	validation_0-aucpr:0.63423
[4]	validation_0-aucpr:0.69332
[5]	validation_0-aucpr:0.70103
[6]	validation_0-aucpr:0.67087
[7]	validation_0-aucpr:0.71055
[8]	validation_0-aucpr:0.62532
[9]	validation_0-aucpr:0.73425
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=depthwise, learning_rate=2.465832945854912, max_depth=10, n_estimators=3125, re

RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=[<xgboost.callback.EarlyStopping object at 0x7fbbae075810>],
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='aucpr',
                                           feature_types=None, gamma=None,
                                           grow_policy=None,
                                           importan...
                   param_distributions={'colsample_bytree': [0.8, 1.0],
                                        'gamma': [0, 0.1, 1],
                                        'grow_policy': ['depthwise',
                                                        'lossguide'],
                                        'learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7fbba01d54d0>,
                                        'max_depth': [None, 3, 5, 7, 10],
                                        'n_estimators': array([ 625,  934, 1397, 2089, 3125]),
                                        'reg_alpha': [0, 0.1, 1, 5],
                                        'reg_lambda': [0.1, 1, 10, 20],
                                        'subsample': [0.8, 1.0]},
                   random_state=42, scoring='accuracy', verbose=2)

In [ ]:
try:
    path = 'classifier.json'
    with open(path, 'r') as file:
        info = json.loads(file.read())
except FileNotFoundError:
    info = {}
    print(info)
    raise ValueError('info is empty')
info

In [ ]:
def save(model):
    x_train,x_test,y_train,y_test = train_test(x,y)
    y_pred = model.predict(x_test)
    name = str(model.estimator)
    i = len(info)
    info.update({name+'_'+str(i+1):{'acc':accuracy_score(y_test,y_pred), 'best_p':model.best_params_}})
    with open(path, 'w') as file:
        file.write(json.dumps(info, default=str))

In [44]:
# save(grid_rf)
# save(grid_gb)
# save(grid_log)
# save(grid_xgb)

In [ ]:
# try:
#     with open('RULC.json', 'r') as file:
#         info = json.loads(file.read())
# except FileNotFoundError:
#     info = {}
# print(info)
# # info.update({'grid_rf':{'acc':accuracy_score(y_test,y_pred_rf),'best_p':grid_rf.best_params_}})
# # # print(info)
# # info.update({'grid_gb':{'acc':accuracy_score(y_test,y_pred_gb),'best_p':grid_gb.best_params_}})
# # # print(info)
# # info.update({'grid_log':{'acc':accuracy_score(y_test,y_pred_log),'best_p':grid_log.best_params_}})
# # print(info)
# info.update({'grid_xgb_2':{'acc':accuracy_score(y_test,y_pred_xgb),'best_p':grid_xgb.best_params_}})
# # print(info)
# info

In [40]:
type(info)

dict

In [41]:
# import json
# with open('RULC.json', 'w') as file:
#     json.dump(info, file, default=str, indent=4)

In [42]:
info

{'RandomForestClassifier(class_weight={0: 1, 1: 7}, random_state=42)_1': {'acc': 0.9286868220632356,
  'best_p': {'n_estimators': 625,
   'min_samples_split': 6,
   'min_samples_leaf': 5,
   'max_samples': 0.8,
   'max_features': 'sqrt',
   'max_depth': 3,
   'criterion': 'gini',
   'bootstrap': True}},
 'RandomForestClassifier(class_weight={0: 1, 1: 7}, random_state=42)_2': {'acc': 0.9286868220632356,
  'best_p': {'n_estimators': 625,
   'min_samples_split': 6,
   'min_samples_leaf': 5,
   'max_samples': 0.8,
   'max_features': 'sqrt',
   'max_depth': 3,
   'criterion': 'gini',
   'bootstrap': True}},
 'RandomForestClassifier(class_weight={0: 1, 1: 7}, random_state=42)_3': {'acc': 0.9286868220632356,
  'best_p': {'n_estimators': 625,
   'min_samples_split': 6,
   'min_samples_leaf': 5,
   'max_samples': 0.8,
   'max_features': 'sqrt',
   'max_depth': 3,
   'criterion': 'gini',
   'bootstrap': True}},
 'GradientBoostingClassifier(n_iter_no_change=10, random_state=42)_4': {'acc': 0.9457